# Perfiles de estudiantes Saber 11 (Colombia, 2018) — 35 variables

**Problema real:** focalizar apoyos educativos según el perfil académico y socioeconómico del estudiante.

**Dataset:** muestra de 1.500 estudiantes reales de **Saber 11° 2018-2** (ICFES). Este notebook lee el CSV desde tu propio repositorio de GitHub (ver instrucciones en la Celda 3) en vez de tenerlo embebido -- sube 'saber11_2018_muestra.csv' a un repo público y pega la URL raw en la Celda 3.

**35 variables:** 16 oficiales del ICFES + 19 derivadas y documentadas explícitamente.

**Algoritmos:** K-Means, DBSCAN (sobre PCA-2D), EM (GMM) y CobWeb (implementado desde cero).

## CELDA 1 — Introducción

In [ ]:
# PROBLEMA: Agrupación de estudiantes colombianos según su desempeño académico
# y contexto socioeconómico, para que una secretaría de educación (o el ICFES)
# pueda FOCALIZAR APOYOS (tutorías, subsidios de conectividad, refuerzo por
# área) en los perfiles de estudiantes que más lo necesitan, en vez de aplicar
# programas genéricos a toda la población estudiantil.
#
# POR QUÉ CLUSTERING: no existe una categoría oficial de "tipo de estudiante"
# -- se busca DESCUBRIR perfiles naturales que combinan desempeño académico
# (puntajes por área) con contexto socioeconómico y hábitos de estudio.
#
# FUENTE: Resultados Saber 11° 2018-2 (ICFES), vía Datos Abiertos Colombia
# (datos.gov.co/Educaci-n/Saber-11-2018-2/m2nt-jw2h). Dataset real y público,
# 494.856 estudiantes en el archivo completo; aquí se usa una MUESTRA
# ALEATORIA de 1.500 estudiantes para que el notebook corra en tiempos
# razonables (esto se declara explícitamente como limitación más adelante).

## CELDA 2 — Imports

In [ ]:
import io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import defaultdict
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.mixture import GaussianMixture
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

## CELDA 3 — Dataset (muestra real de 1500 estudiantes, leída desde tu repositorio de GitHub)

In [ ]:
# PASO PREVIO (hazlo una sola vez):
# 1. Crea un repositorio en GitHub (puede ser público o privado -- si es
#    privado, este método de descarga directa no funcionará y tendrías que
#    usar un token; lo más simple para un taller académico es dejarlo público).
# 2. Sube el archivo 'saber11_2018_muestra.csv' a ese repositorio.
# 3. Entra al archivo subido en GitHub, dale clic a "Raw", y copia esa URL
#    (se ve así: https://raw.githubusercontent.com/tu-usuario/tu-repo/main/saber11_2018_muestra.csv)
# 4. Pega esa URL abajo, reemplazando el texto de ejemplo.

GITHUB_RAW_URL = "https://raw.githubusercontent.com/TU-USUARIO/TU-REPO/main/saber11_2018_muestra.csv"

df = pd.read_csv(GITHUB_RAW_URL)
print("Total de estudiantes en esta muestra:", len(df))
print("Variables originales (16, del dataset oficial ICFES):", list(df.columns))

## CELDA 4 — Diccionario de las 16 variables originales

In [ ]:
descripcion_originales = {
    "ESTU_GENERO": "Género del estudiante (0/1)",
    "ESTU_DEPTO_RESIDE": "Departamento de residencia (código 1-33)",
    "FAMI_ESTRATOVIVIENDA": "Estrato socioeconómico de la vivienda (0-6)",
    "FAMI_TIENEINTERNET": "El hogar tiene internet (0=no, 1=sí)",
    "FAMI_TIENESERVICIOTV": "El hogar tiene servicio de TV (0=no, 1=sí)",
    "FAMI_TIENECOMPUTADOR": "El hogar tiene computador (0=no, 1=sí)",
    "FAMI_SITUACIONECONOMICA": "Percepción de la situación económica familiar (0-2)",
    "ESTU_DEDICACIONLECTURADIARIA": "Minutos diarios dedicados a lectura",
    "ESTU_DEDICACIONINTERNET": "Minutos diarios dedicados a internet",
    "COLE_AREA_UBICACION": "Zona del colegio (0=urbana, 1=rural)",
    "PUNT_LECTURA_CRITICA": "Puntaje en Lectura Crítica (0-100)",
    "PUNT_MATEMATICAS": "Puntaje en Matemáticas (0-100)",
    "PUNT_C_NATURALES": "Puntaje en Ciencias Naturales (0-100)",
    "PUNT_SOCIALES_CIUDADANAS": "Puntaje en Sociales y Ciudadanas (0-100)",
    "PUNT_INGLES": "Puntaje en Inglés (0-100)",
    "PUNT_GLOBAL": "Puntaje global (0-500)",
}
for k, v in descripcion_originales.items():
    print(f"{k:30s}: {v}")

## CELDA 5 — Ingeniería de variables: de 16 a 35 (19 variables derivadas, documentadas)

In [ ]:
# El profesor pide 35 variables. El dataset oficial trae 16; se añaden 19 MÁS
# calculadas a partir de esas mismas 16 (no inventadas), cada una con valor
# analítico propio para el problema de "perfiles de estudiante":
areas = ["PUNT_LECTURA_CRITICA", "PUNT_MATEMATICAS", "PUNT_C_NATURALES",
         "PUNT_SOCIALES_CIUDADANAS", "PUNT_INGLES"]

df["PROMEDIO_AREAS"] = df[areas].mean(axis=1)                          # 17
df["DESVIACION_AREAS"] = df[areas].std(axis=1)                         # 18
df["BRECHA_MEJOR_PEOR"] = df[areas].max(axis=1) - df[areas].min(axis=1)  # 19
df["MEJOR_AREA"] = df[areas].idxmax(axis=1)                            # 20 (categórica)
df["PEOR_AREA"] = df[areas].idxmin(axis=1)                             # 21 (categórica)
df["TIENE_INTERNET_Y_TV"] = ((df["FAMI_TIENEINTERNET"] == 1) &
                              (df["FAMI_TIENESERVICIOTV"] == 1)).astype(int)  # 22
df["INDICE_RECURSOS_HOGAR"] = (df["FAMI_TIENEINTERNET"] + df["FAMI_TIENESERVICIOTV"]
                                + df["FAMI_TIENECOMPUTADOR"])           # 23
df["DEDICACION_TOTAL_MIN"] = (df["ESTU_DEDICACIONLECTURADIARIA"]
                               + df["ESTU_DEDICACIONINTERNET"])        # 24
df["ZONA_RURAL"] = df["COLE_AREA_UBICACION"]                           # 25
df["NIVEL_INGLES"] = pd.cut(df["PUNT_INGLES"], bins=[-1, 40, 60, 100],
                             labels=["bajo", "medio", "alto"])          # 26

# --- Variables 27-35: z-scores por área, banda oficial de desempeño, percentil,
#     razón verbal/cuantitativa y agrupación de estrato ---
for area in areas:
    df[f"Z_{area.replace('PUNT_', '')}"] = (df[area] - df[area].mean()) / df[area].std()  # 27-31 (5 columnas)

# Bandas de desempeño global -- el ICFES clasifica oficialmente el puntaje
# global en 4 niveles (Bajo/Medio/Alto/Superior); se reproduce esa lógica
# aproximada según los cortes históricos publicados por el ICFES.
df["NIVEL_DESEMPENO_GLOBAL"] = pd.cut(
    df["PUNT_GLOBAL"], bins=[-1, 250, 300, 350, 500],
    labels=["Bajo", "Medio", "Alto", "Superior"]
)                                                                        # 32 (categórica)

df["PERCENTIL_GLOBAL"] = df["PUNT_GLOBAL"].rank(pct=True) * 100         # 33
df["RATIO_LECTURA_MATEMATICAS"] = df["PUNT_LECTURA_CRITICA"] / df["PUNT_MATEMATICAS"].replace(0, np.nan)  # 34
df["GRUPO_ESTRATO"] = pd.cut(
    df["FAMI_ESTRATOVIVIENDA"], bins=[-1, 1, 3, 6],
    labels=["bajo", "medio", "alto"]
)                                                                        # 35 (categórica)

print(f"\nTotal de variables ahora: {df.shape[1]} "
      f"(16 originales + 19 derivadas = 35 variables de análisis)")

## CELDA 6 — Preparación de datos para clustering

In [ ]:
# Variables NUMÉRICAS usadas para K-Means / DBSCAN / EM (se excluyen las
# categóricas de texto MEJOR_AREA, PEOR_AREA, NIVEL_INGLES, NIVEL_DESEMPENO_GLOBAL,
# GRUPO_ESTRATO, que se usan en CobWeb en su lugar).
features_numericas = [
    "ESTU_GENERO", "ESTU_DEPTO_RESIDE", "FAMI_ESTRATOVIVIENDA", "FAMI_TIENEINTERNET",
    "FAMI_TIENESERVICIOTV", "FAMI_TIENECOMPUTADOR", "FAMI_SITUACIONECONOMICA",
    "ESTU_DEDICACIONLECTURADIARIA", "ESTU_DEDICACIONINTERNET", "COLE_AREA_UBICACION",
    "PUNT_LECTURA_CRITICA", "PUNT_MATEMATICAS", "PUNT_C_NATURALES",
    "PUNT_SOCIALES_CIUDADANAS", "PUNT_INGLES", "PUNT_GLOBAL",
    "PROMEDIO_AREAS", "DESVIACION_AREAS", "BRECHA_MEJOR_PEOR",
    "TIENE_INTERNET_Y_TV", "INDICE_RECURSOS_HOGAR", "DEDICACION_TOTAL_MIN", "ZONA_RURAL",
    "Z_LECTURA_CRITICA", "Z_MATEMATICAS", "Z_C_NATURALES", "Z_SOCIALES_CIUDADANAS", "Z_INGLES",
    "PERCENTIL_GLOBAL", "RATIO_LECTURA_MATEMATICAS",
]
print(f"\n{len(features_numericas)} variables numéricas para K-Means/DBSCAN/EM")

X_raw = df[features_numericas].values
scaler = StandardScaler()
X = scaler.fit_transform(X_raw)

## CELDA 7 — K-MEANS con justificación automática de k

In [ ]:
inertias = []
k_range = list(range(2, 11))
for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X)
    inertias.append(km.inertia_)

plt.figure(figsize=(6, 4))
plt.plot(k_range, inertias, marker="o")
plt.xlabel("k")
plt.ylabel("Inercia (WCSS)")
plt.title("Método del codo - K-Means (estudiantes)")
plt.tight_layout()
plt.savefig("01_codo_kmeans.png", dpi=150)
plt.show()


def encontrar_codo(x_vals, y_vals):
    x = np.array(x_vals, dtype=float)
    y = np.array(y_vals, dtype=float)
    x_n = (x - x.min()) / (x.max() - x.min())
    y_n = (y - y.min()) / (y.max() - y.min())
    p1, p2 = np.array([x_n[0], y_n[0]]), np.array([x_n[-1], y_n[-1]])
    d = p2 - p1

    def cross2d(a, b):
        return a[0] * b[1] - a[1] * b[0]

    dists = [abs(cross2d(d, p1 - np.array([xi, yi]))) / np.linalg.norm(d)
             for xi, yi in zip(x_n, y_n)]
    return int(x[int(np.argmax(dists))])


K_OPTIMO = encontrar_codo(k_range, inertias)
print(f"K sugerido por el método del codo: k = {K_OPTIMO}")

kmeans = KMeans(n_clusters=K_OPTIMO, random_state=42, n_init=10)
df["cluster_kmeans"] = kmeans.fit_predict(X)

## CELDA 8 — DBSCAN (sobre proyección PCA-2D, con calibración de eps)

In [ ]:
# Con 30 variables numéricas, DBSCAN sufre la maldición de la dimensionalidad;
# se aplica sobre las 2 componentes principales, igual que en el proyecto de
# pacientes.
X_2d = PCA(n_components=2, random_state=42).fit_transform(X)

MIN_SAMPLES = 8
nn = NearestNeighbors(n_neighbors=MIN_SAMPLES).fit(X_2d)
distancias, _ = nn.kneighbors(X_2d)
k_distancias = np.sort(distancias[:, -1])

plt.figure(figsize=(6, 4))
plt.plot(k_distancias)
plt.xlabel("Puntos ordenados")
plt.ylabel(f"Distancia al {MIN_SAMPLES}-ésimo vecino (PCA-2D)")
plt.title("Gráfico de k-distancia para elegir eps de DBSCAN")
plt.tight_layout()
plt.savefig("02_kdistancia_dbscan.png", dpi=150)
plt.show()

EPS = float(np.percentile(k_distancias, 80))
print(f"eps elegido (percentil 80, espacio PCA-2D): {EPS:.3f}")

dbscan = DBSCAN(eps=EPS, min_samples=MIN_SAMPLES)
df["cluster_dbscan"] = dbscan.fit_predict(X_2d)
n_ruido = (df["cluster_dbscan"] == -1).sum()
n_clusters_db = df["cluster_dbscan"].nunique() - (1 if -1 in df["cluster_dbscan"].values else 0)
print(f"DBSCAN: {n_clusters_db} clusters, {n_ruido} estudiantes atípicos ({100*n_ruido/len(df):.1f}%)")

## CELDA 9 — EM (implementado mediante Gaussian Mixture Model)

In [ ]:
gmm = GaussianMixture(n_components=K_OPTIMO, random_state=42)
df["cluster_gmm"] = gmm.fit_predict(X)

## CELDA 10 — CobWeb (sobre variables categóricas: incluye las 3 nuevas: MEJOR_AREA, PEOR_AREA, NIVEL_INGLES)

In [ ]:
class CobwebNode:
    __slots__ = ["children", "counts", "n", "instances"]

    def __init__(self):
        self.children = []
        self.counts = defaultdict(lambda: defaultdict(int))
        self.n = 0
        self.instances = []

    def add(self, instance, idx):
        self.n += 1
        self.instances.append(idx)
        for attr, val in instance.items():
            self.counts[attr][val] += 1

    def attr_value_prob(self, attr, val):
        return 0.0 if self.n == 0 else self.counts[attr].get(val, 0) / self.n


def category_utility(parent, children):
    if not children:
        return -1e9
    total_n = parent.n
    k = len(children)
    cu_sum = 0.0
    for child in children:
        if child.n == 0:
            continue
        pc = child.n / total_n
        inner = 0.0
        for attr in parent.counts.keys():
            for val in set(parent.counts[attr].keys()):
                inner += child.attr_value_prob(attr, val) ** 2 - parent.attr_value_prob(attr, val) ** 2
        cu_sum += pc * inner
    return cu_sum / k


_instances_ref = None


def cobweb_incorporate(root, instance, idx):
    node = root
    while True:
        node.add(instance, idx)
        if not node.children:
            if node.n == 1:
                return
            prev_indices = node.instances[:-1]
            if prev_indices:
                child_prev = CobwebNode()
                for j in prev_indices:
                    child_prev.add(_instances_ref[j], j)
                node.children.append(child_prev)
            child_new = CobwebNode()
            child_new.add(instance, idx)
            node.children.append(child_new)
            return

        best_cu, best_child = -1e18, None
        for child in node.children:
            trial = CobwebNode()
            trial.n = child.n + 1
            trial.counts = defaultdict(lambda: defaultdict(int),
                                        {a: dict(v) for a, v in child.counts.items()})
            for attr, val in instance.items():
                trial.counts[attr][val] = trial.counts[attr].get(val, 0) + 1
            others = [c for c in node.children if c is not child]
            cu = category_utility(node, others + [trial])
            if cu > best_cu:
                best_cu, best_child = cu, child

        new_child = CobwebNode()
        new_child.add(instance, idx)
        cu_new = category_utility(node, node.children + [new_child])

        if cu_new >= best_cu:
            node.children.append(new_child)
            return
        else:
            node = best_child


def extract_clusters(root, level=1):
    frontier = [root]
    for _ in range(level):
        nf = []
        for n in frontier:
            nf.extend(n.children) if n.children else nf.append(n)
        frontier = nf
    labels = {}
    for cid, node in enumerate(frontier):
        for idx in node.instances:
            labels[idx] = cid
    return labels, len(frontier)


def run_cobweb(list_of_dicts):
    global _instances_ref
    _instances_ref = list_of_dicts
    root = CobwebNode()
    for i, inst in enumerate(list_of_dicts):
        cobweb_incorporate(root, inst, i)
    return root


cols_cobweb = ["FAMI_ESTRATOVIVIENDA", "FAMI_SITUACIONECONOMICA", "COLE_AREA_UBICACION",
               "MEJOR_AREA", "PEOR_AREA", "NIVEL_INGLES", "NIVEL_DESEMPENO_GLOBAL", "GRUPO_ESTRATO"]
instancias_cw = df[cols_cobweb].astype(str).to_dict("records")

root_cw = run_cobweb(instancias_cw)
labels_cw, n_clusters_cw = extract_clusters(root_cw, level=1)
df["cluster_cobweb"] = df.index.map(labels_cw)
print(f"CobWeb: {n_clusters_cw} clusters (sobre {len(cols_cobweb)} variables categóricas, n={len(df)})")

## CELDA 11 — MÉTRICAS Y TABLA COMPARATIVA

In [ ]:
def metricas_numericas(labels, X):
    labels = np.array(labels)
    mask = labels != -1
    if mask.sum() < 2 or len(set(labels[mask])) < 2:
        return None, None, None
    return (silhouette_score(X[mask], labels[mask]),
            davies_bouldin_score(X[mask], labels[mask]),
            calinski_harabasz_score(X[mask], labels[mask]))


sil_km, db_km, ch_km = metricas_numericas(df["cluster_kmeans"], X)
sil_db, db_db, ch_db = metricas_numericas(df["cluster_dbscan"], X_2d)
sil_em, db_em, ch_em = metricas_numericas(df["cluster_gmm"], X)

tabla_comparacion = pd.DataFrame([
    {"Algoritmo": "K-Means", "N_clusters": df["cluster_kmeans"].nunique(), "Ruido_%": 0.0,
     "Silhouette": sil_km, "Davies-Bouldin": db_km},
    {"Algoritmo": "DBSCAN", "N_clusters": n_clusters_db, "Ruido_%": round(100 * n_ruido / len(df), 1),
     "Silhouette": sil_db, "Davies-Bouldin": db_db},
    {"Algoritmo": "EM (GMM)", "N_clusters": df["cluster_gmm"].nunique(), "Ruido_%": 0.0,
     "Silhouette": sil_em, "Davies-Bouldin": db_em},
    {"Algoritmo": "CobWeb", "N_clusters": n_clusters_cw, "Ruido_%": np.nan,
     "Silhouette": np.nan, "Davies-Bouldin": np.nan},
])
print("\n" + "=" * 70)
print("TABLA COMPARATIVA")
print("=" * 70)
print(tabla_comparacion.to_string(index=False))

## CELDA 12 — VISUALIZACIÓN

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, col, titulo in zip(axes, ["cluster_kmeans", "cluster_dbscan", "cluster_gmm"],
                           ["K-Means", "DBSCAN", "EM (GMM)"]):
    ax.scatter(X_2d[:, 0], X_2d[:, 1], c=df[col], cmap="tab10", s=20)
    ax.set_title(titulo)
    ax.set_xlabel("Componente principal 1")
    ax.set_ylabel("Componente principal 2")
plt.suptitle("Clusters proyectados en 2D (PCA) -- 30 variables numéricas reducidas a 2 ejes")
plt.tight_layout()
plt.savefig("03_clusters_pca.png", dpi=150)
plt.show()

# CobWeb: gráfica de barras (variables categóricas, no aplica proyección espacial)
cw_tab = pd.crosstab(df["cluster_cobweb"], df["NIVEL_INGLES"])
fig, ax = plt.subplots(figsize=(7, 4))
cw_tab.plot(kind="bar", stacked=True, ax=ax, colormap="Set2")
ax.set_xlabel("Cluster CobWeb")
ax.set_ylabel("N° de estudiantes")
ax.set_title("CobWeb — nivel de inglés por cluster")
plt.tight_layout()
plt.savefig("04_barras_cobweb.png", dpi=150)
plt.show()

## CELDA 13 — PERFIL DE CADA CLUSTER (K-Means)

In [ ]:
cols_perfil = ["PUNT_GLOBAL", "PROMEDIO_AREAS", "DESVIACION_AREAS", "FAMI_ESTRATOVIVIENDA",
               "INDICE_RECURSOS_HOGAR", "DEDICACION_TOTAL_MIN", "ZONA_RURAL"]
perfil = df.groupby("cluster_kmeans")[cols_perfil].mean().round(1)
perfil["n_estudiantes"] = df.groupby("cluster_kmeans").size()
print("\nPerfil promedio por cluster (K-Means):")
print(perfil)

## CELDA 14 — INTERPRETACIÓN Y DECISIONES (generadas de los resultados reales)

In [ ]:
cluster_mejor_desempeno = perfil["PUNT_GLOBAL"].idxmax()
cluster_peor_desempeno = perfil["PUNT_GLOBAL"].idxmin()

print("=" * 70)
print("INTERPRETACIÓN Y DECISIONES")
print("=" * 70)
print(f"""
1) CLUSTER DE MEJOR DESEMPEÑO: cluster {cluster_mejor_desempeno}
   - Puntaje global promedio: {perfil.loc[cluster_mejor_desempeno, 'PUNT_GLOBAL']:.0f}
   - Estrato promedio: {perfil.loc[cluster_mejor_desempeno, 'FAMI_ESTRATOVIVIENDA']:.1f}
   - Índice de recursos del hogar: {perfil.loc[cluster_mejor_desempeno, 'INDICE_RECURSOS_HOGAR']:.1f} de 3
   >> DECISIÓN: estos estudiantes podrían orientarse hacia programas de becas
      universitarias de alto perfil, en vez de refuerzo académico básico.

2) CLUSTER DE MENOR DESEMPEÑO: cluster {cluster_peor_desempeno}
   - Puntaje global promedio: {perfil.loc[cluster_peor_desempeno, 'PUNT_GLOBAL']:.0f}
   - Estrato promedio: {perfil.loc[cluster_peor_desempeno, 'FAMI_ESTRATOVIVIENDA']:.1f}
   - Índice de recursos del hogar: {perfil.loc[cluster_peor_desempeno, 'INDICE_RECURSOS_HOGAR']:.1f} de 3
   - % en zona rural: {perfil.loc[cluster_peor_desempeno, 'ZONA_RURAL']*100:.0f}%
   >> DECISIÓN: este es el perfil que debería priorizarse para programas de
      conectividad (si el índice de recursos es bajo) y tutorías focalizadas,
      en línea con lo que ya reporta la literatura sobre brechas educativas
      urbano-rurales en Colombia.

3) SOBRE LA DISPERSIÓN ENTRE ÁREAS (variable derivada DESVIACION_AREAS): un
   cluster con alta desviación entre sus puntajes por área indica estudiantes
   "desbalanceados" (fuertes en unas materias, débiles en otras) -- estos se
   benefician más de refuerzo dirigido a su área más débil que de un programa
   genérico que repase todas las materias por igual.

NOTA IMPORTANTE: el desempeño está correlacionado con el contexto
socioeconómico en estos datos, pero el clustering no prueba causalidad -- solo
identifica el patrón para poder actuar sobre él.
""")

## CELDA 15 — LIMITACIONES

In [ ]:
print("=" * 70)
print("LIMITACIONES")
print("=" * 70)
print(f"""
- Se usó una MUESTRA ALEATORIA de {len(df)} estudiantes del total de 494.856
  registros del dataset completo Saber 11° 2018-2 -- una muestra distinta
  podría arrojar centroides ligeramente diferentes.
- De las 35 variables analizadas, 16 son las originales del dataset oficial
  del ICFES y 19 son derivadas (calculadas a partir de esas 16) para enriquecer
  el análisis: promedios, desviaciones, z-scores por área, bandas oficiales de
  desempeño, percentil e índices compuestos. Esto se documenta explícitamente
  porque no es lo mismo que 35 variables originales independientes.
- K-Means y EM requieren fijar k; se usó k={K_OPTIMO} por el método del codo.
- DBSCAN corrió sobre una proyección PCA-2D (no las 30 variables numéricas
  completas) por su sensibilidad a la dimensionalidad.
- El dataset es de 2018 -- los patrones podrían haber cambiado (ej. tras la
  pandemia, el acceso a internet/conectividad rural cambió significativamente).
- El clustering muestra correlación entre contexto socioeconómico y desempeño,
  no relaciones causales -- no reemplaza un diagnóstico pedagógico individual.
- Sería recomendable validar estos perfiles con datos más recientes (2023-2024)
  y con una muestra estratificada por departamento, ya que esta muestra
  aleatoria simple podría sobre-representar departamentos con más estudiantes.
""")

df.to_csv("estudiantes_clustering_resultado.csv", index=False)
print("Archivo 'estudiantes_clustering_resultado.csv' generado.")

## CELDA 16 — CONCLUSIONES

In [ ]:
mejor_silhouette = max(
    [("K-Means", sil_km), ("DBSCAN", sil_db), ("EM (GMM)", sil_em)],
    key=lambda t: t[1] if t[1] is not None else -1,
)

print("=" * 70)
print("CONCLUSIONES")
print("=" * 70)
print(f"""
1. Sobre una muestra de {len(df)} estudiantes reales de Saber 11° 2018-2, con
   35 variables (16 oficiales + 19 derivadas), los cuatro algoritmos lograron
   identificar perfiles de estudiante diferenciados por desempeño y contexto.

2. K-Means y EM, con k={K_OPTIMO} determinado por el método del codo, separaron un
   cluster de mejor desempeño (cluster {cluster_mejor_desempeno}, puntaje global
   promedio {perfil.loc[cluster_mejor_desempeno, 'PUNT_GLOBAL']:.0f}) de uno de menor
   desempeño (cluster {cluster_peor_desempeno}, promedio {perfil.loc[cluster_peor_desempeno, 'PUNT_GLOBAL']:.0f}),
   una diferencia asociada también a diferencias en estrato socioeconómico y
   recursos del hogar.

3. DBSCAN, calibrado con el gráfico de k-distancia, detectó {n_clusters_db} grupos
   con un {100*n_ruido/len(df):.1f}% de estudiantes atípicos -- perfiles que no encajan
   claramente en ningún patrón dominante de la muestra.

4. CobWeb, trabajando sobre variables categóricas (estrato, zona, nivel de
   inglés, mejor/peor área), encontró {n_clusters_cw} grupos, mostrando que incluso
   simplificando a categorías se mantienen patrones reconocibles de perfil
   estudiantil.

5. De los tres algoritmos comparables numéricamente, {mejor_silhouette[0]} obtuvo
   el mejor silhouette ({mejor_silhouette[1]:.3f}), pero la decisión de política
   educativa debería considerar los cuatro en conjunto: la partición completa
   de K-Means/EM es útil para asignar recursos a TODOS los estudiantes, mientras
   que el ruido de DBSCAN señala casos individuales que merecen atención aparte.
""")
print("=== FIN DEL ANÁLISIS ===")